# Paper 3 — remaining work (Colab)

Runs what is left after the 2026-09-19 pipeline run, in dependency order.

| Step | Needs GPU? | Time | Produces |
|---|---|---|---|
| 1. C7 LOSO DEV tuning | **yes** | 1–2 h | `loso_dev_selected_hyperparams.json` |
| 2. LOSO rerun with those settings | **yes** | ~1 h | corrected LOSO numbers |
| 3. CDHW contribution, incl. NeuralCQR seeds | yes (faster) | 30–60 min | replaces the retired O1 |
| 4. Rolling origin (optional rerun) | no | ~50 min | `rolling_origin_rows.csv` |
| 5. RO1–RO5 objectives | no | seconds | `objectives_RO_report.json` |

**Expect the LOSO macro R² to FALL below 0.6976 after step 2.** The current value
was produced with hyperparameters chosen by watching held-out performance; an
honest re-derivation removes that advantage. A drop is the correct outcome.


## 1. Project into place

Same folder as the master runner: it must contain `code/` and `master_dataset/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/Paper3_MASTER'

import os, sys, importlib
os.environ['PAPER3_DATA_SOURCE'] = 'master'
CODE_DIR = os.path.join(PROJECT_ROOT, 'code')
assert os.path.isdir(CODE_DIR), f'code/ not found at {CODE_DIR}'
os.chdir(PROJECT_ROOT); sys.path.insert(0, CODE_DIR)

import torch, config
importlib.reload(config)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
print('data source:', config.DATA_SOURCE)

from frozen_protocol import protocol_hash, verify_against_config
print('protocol hash:', protocol_hash(), '| config matches:', not verify_against_config())

### Files that must be synced

The new modules, plus the artefacts steps 3–5 read. If `rolling_origin_rows.csv`
is missing you can regenerate it with step 4 instead of uploading it.

In [ ]:
need = ['code/loso_dev_tuning.py', 'code/objectives_ro.py', 'code/cdhw_contribution.py',
        'code/rolling_origin.py', 'code/baselines.py', 'code/decision_impact.py',
        'code/frozen_protocol.py',
        'outputs_master/predictions/loso_predictions.csv',
        'outputs_master/reports/loso_fold_metrics.csv']
optional = ['outputs_diagnostics/reports/rolling_origin_rows.csv',
            'outputs_diagnostics/reports/rolling_origin_raw.csv',
            'outputs_diagnostics/reports/decision_impact_by_stratum.csv']
for f in need:
    print(('OK   ' if os.path.exists(f) else 'MISS '), f)
for f in optional:
    print(('OK   ' if os.path.exists(f) else 'opt  '), f, '' if os.path.exists(f) else '(regenerate with step 4)')

## 2. C7 — re-derive LOSO hyperparameters on DEV

Six folds x four pre-declared configurations. The held-out state is never read;
each fold trains on its own FIT rows, early-stops on the FIT carve-out, and is
scored on its own DEV partition.

In [ ]:
!python -u code/loso_dev_tuning.py --seeds 1 2>&1 | grep -vE "^INFO|Epoch" | tail -40

### Apply the selection

Flip the flag, bump the protocol version (the hash will change — that is the
mechanism working), then rerun the pipeline so the LOSO numbers come from
DEV-selected settings. Each fold will stamp `hyperparameter_source: DEV-selected`.

In [ ]:
import re, pathlib
p = pathlib.Path('code/config.py'); s = p.read_text(encoding='utf-8')
s = s.replace('LOSO_USE_DEV_SELECTED_HP: bool = False', 'LOSO_USE_DEV_SELECTED_HP: bool = True')
p.write_text(s, encoding='utf-8')

f = pathlib.Path('code/frozen_protocol.py'); t = f.read_text(encoding='utf-8')
t = t.replace('PROTOCOL_VERSION = "1.2.0-2026-09-18"', 'PROTOCOL_VERSION = "2.0.0-C7-applied"')
f.write_text(t, encoding='utf-8')

import importlib, config, frozen_protocol
importlib.reload(config); importlib.reload(frozen_protocol)
print('LOSO_USE_DEV_SELECTED_HP =', config.LOSO_USE_DEV_SELECTED_HP)
print('new protocol hash:', frozen_protocol.protocol_hash())
print('mismatches:', frozen_protocol.verify_against_config())

In [ ]:
import subprocess, sys
proc = subprocess.Popen([sys.executable, '-u', 'code/main.py'], stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1,
                        env=dict(os.environ, PAPER3_DATA_SOURCE='master'))
KEY = ('Frozen protocol', 'hyperparameters:', 'LOSO Fold', '-> RMSE=', 'LOSO-CV',
       'Traceback', 'ERROR')
for line in proc.stdout:
    if any(k in line for k in KEY):
        print(line.rstrip())
print('exit:', proc.wait())

## 3. CDHW contribution across models (replaces the retired O1)

In [ ]:
!python -u code/cdhw_contribution.py --tree-seeds 3 --neural-seeds 5 2>&1 | grep -vE "^INFO|Epoch" | tail -30

## 4. Rolling origin (only if `rolling_origin_rows.csv` is missing)

In [ ]:
# !python -u code/rolling_origin.py --first 2008 --last 2023 --model lgbm 2>&1 | tail -30

## 5. RO1–RO5 objectives

Reads artefacts only; nothing is refitted.

In [ ]:
!python -u code/objectives_ro.py 2>&1 | tail -40

## 6. Compare before and after C7

In [ ]:
import pandas as pd
fm = pd.read_csv('outputs_master/reports/loso_fold_metrics.csv')
print('LOSO macro R2 now :', round(fm.r_squared.mean(), 4), '   (was 0.6976 with config defaults)')
print('hyperparameter_source:', fm.get('hyperparameter_source', pd.Series(['absent'])).unique().tolist())
print(fm[['state', 'r_squared', 'picp']].to_string(index=False))

## 7. Save results back

In [ ]:
import shutil, datetime
stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M')
for folder in ('outputs_master', 'outputs_diagnostics'):
    if os.path.isdir(folder):
        shutil.make_archive(f'/content/paper3_{folder}_{stamp}', 'zip', folder)
        print('wrote', f'/content/paper3_{folder}_{stamp}.zip')